# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR² clinicopathological dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source

The dataset is described by a Croissant schema, accessible at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed in your environment
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets and fields, referencing all using their Croissant `@id`s.

In [ ]:
# Listing available record sets by their @id
record_sets = list(dataset.metadata.record_sets)
print("Available Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs['name']}")

In [ ]:
# For each record set, list its fields and columns by @id, if available
for rs in record_sets:
    print(f"\nRecord Set: {rs['name']} (@id: {rs['@id']})")
    if 'fields' in rs:
        print("  Fields:")
        for f in rs['fields']:
            # Field is a dict if hydrated, else a reference
            if isinstance(f, dict):
                print(f"    - {f['name']} (@id: {f['@id']})")
            else:
                print(f"    - @id: {f}")
    if 'columns' in rs:
        print("  Columns:")
        for c in rs['columns']:
            if isinstance(c, dict):
                print(f"    - {c['name']} (@id: {c['@id']})")
            else:
                print(f"    - @id: {c}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis, referencing it by `@id`.

In [ ]:
# Choose a main record set (usually with the tabular/subject data)
# For this dataset: 
# - Most likely, the only or main record set will have an @id like 'https://api.app.sen.science/frontiers/7862866/6ebff3ad-67de-4e43-882b-1b1d64979b70' 
# To find the correct @id, inspect the output from the previous overview step.

# For demonstration, let's assign the variable to the actual @id of the record set found in the previous cell's output.
main_record_set_id = '<INSERT_MAIN_RECORD_SET_ID_HERE>'  # Replace with real @id from overview

# If there are multiple record sets, collect their @id's
record_set_ids = [main_record_set_id]
dataframes = {}

for rs_id in record_set_ids:
    print(f"\nLoading records from record set @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records, columns: {df.columns.tolist()}")

# Show columns of main DataFrame and a preview
print("\nColumns available:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering, normalizing, and grouping by categorical attributes. All fields referenced by their Croissant `@id`s.

In [ ]:
# Select a numeric field for analysis by its @id
# Use code from the overview output to find available numeric (@type: Integer/Float) fields. For demonstration, example @id shown below.
numeric_field_id = '<INSERT_NUMERIC_FIELD_ID_HERE>'  # e.g., '@id' for Age or similar numeric column
group_field_id = '<INSERT_GROUP_FIELD_ID_HERE>'      # e.g., '@id' for Sex, MSI status, or group variable
df = dataframes[main_record_set_id]

# Filtering for numeric field > a threshold
threshold = 50
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
else:
    print(f"Column {numeric_field_id} not found.")
    filtered_df = df.copy()

print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
if numeric_field_id in filtered_df.columns and not filtered_df.empty:
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
    ) / filtered_df[numeric_field_id].astype(float).std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping by categorical field
if group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
    display(grouped_df)
else:
    print(f"Group field {group_field_id} not found.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. All axes and references should use the column Croissant `@id`, not labels.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if the required @id columns are available
if numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna().astype(float), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

if group_field_id in df.columns and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

We have demonstrated how to explore the FAIR² dataset using the `mlcroissant` Python library, referencing every entity by its Croissant `@id`. You can continue by:

- Examining additional record sets and fields
- Applying more complex data analysis or ML pipelines
- Citing this dataset using metadata in your downstream applications

For more information, explore the [Croissant specification](https://mlcommons.org/croissant/) and the [mlcroissant API docs](https://mlcroissant.readthedocs.io/).